# AutoML Tabular Workflow Pipelines with Modern Python Dependencies

This notebook demonstrates training classification models on Google Cloud Vertex AI using AutoML Tabular Workflows.
It updates the official Google Cloud sample notebook to modern Python package versions and APIs using `google-cloud-pipeline-components>=2.22.0` (`google_cloud_pipeline_components.v1.automl.tabular`).

### Objectives
1. Configure and launch a customized AutoML Tabular training pipeline on the Bank Marketing dataset.
2. Extract the hyperparameter tuning result artifact from Stage 1.
3. Launch a Skip Architecture Search AutoML Tabular pipeline reusing the architecture tuning results to save time and cost.

In [ ]:
import os

from google.cloud import aiplatform

from tabflows import (
    TabularPipelineConfig,
    build_automl_tabular_pipeline,
    build_skip_architecture_search_pipeline,
)

print("Libraries imported successfully.")

In [ ]:
PROJECT_ID = os.getenv("GCP_PROJECT", "your-project-id")
LOCATION = "us-central1"
BUCKET_URI = f"gs://your-bucket-name-{PROJECT_ID}-unique"

config = TabularPipelineConfig(
    project_id=PROJECT_ID,
    location=LOCATION,
    bucket_uri=BUCKET_URI,
    target_column="deposit",
    prediction_type="classification",
    optimization_objective="minimize-log-loss",
    data_source_csv_filenames="gs://cloud-samples-data/vertex-ai/tabular-workflows/datasets/bank-marketing/train.csv",
)

print(f"Pipeline Root DIR: {config.root_dir}")
print(f"Transform Config Path: {config.transform_config_path}")

In [ ]:
# Example: Extract tuning result artifact URI after job execution:
# task_details = job.gca_resource.job_detail.task_details
# stage_1_task = get_task_detail(task_details, "automl-tabular-stage-1-tuner")
# stage_1_tuning_result_artifact_uri = (
#     stage_1_task.outputs["tuning_result_output"].artifacts[0].uri
# )

stage_1_tuning_result_artifact_uri = f"{config.root_dir}/tuning_result_artifact"

skip_template_path, skip_parameter_values = build_skip_architecture_search_pipeline(
    config=config,
    stage_1_tuning_result_artifact_uri=stage_1_tuning_result_artifact_uri,
)

skip_job_id = "automl-tabular-skip-search-01"
skip_job = aiplatform.PipelineJob(
    display_name=skip_job_id,
    location=config.location,
    template_path=skip_template_path,
    job_id=skip_job_id,
    pipeline_root=config.root_dir,
    parameter_values=skip_parameter_values,
    enable_caching=False,
)

print(f"Skip Architecture Search PipelineJob initialized: {skip_job.display_name}")

In [ ]:
# Build pipeline template and parameters
template_path, parameter_values = build_automl_tabular_pipeline(config)

job_id = "automl-tabular-run-01"

aiplatform.init(project=config.project_id, location=config.location)

job = aiplatform.PipelineJob(
    display_name=job_id,
    location=config.location,
    template_path=template_path,
    job_id=job_id,
    pipeline_root=config.root_dir,
    parameter_values=parameter_values,
    enable_caching=False,
)

# To run job: job.run()
print(f"PipelineJob initialized successfully: {job.display_name}")

## Skip Architecture Search Pipeline

Reusing the hyperparameter tuning result from the stage-1 tuner task reduces training time and cost.

Extract the tuning result artifact URI from `automl-tabular-stage-1-tuner` and pass it to `build_skip_architecture_search_pipeline`.

In [ ]:
# Example: Extract tuning result artifact URI after job execution
# pipeline_task_details = job.gca_resource.job_detail.task_details
# stage_1_tuner_task = get_task_detail(pipeline_task_details, "automl-tabular-stage-1-tuner")
# stage_1_tuning_result_artifact_uri =
# stage_1_tuner_task.outputs["tuning_result_output"].artifacts[0].uri

stage_1_tuning_result_artifact_uri = f"{config.root_dir}/tuning_result_artifact"

skip_template_path, skip_parameter_values = build_skip_architecture_search_pipeline(
    config=config,
    stage_1_tuning_result_artifact_uri=stage_1_tuning_result_artifact_uri,
)

skip_job_id = "automl-tabular-skip-search-01"
skip_job = aiplatform.PipelineJob(
    display_name=skip_job_id,
    location=config.location,
    template_path=skip_template_path,
    job_id=skip_job_id,
    pipeline_root=config.root_dir,
    parameter_values=skip_parameter_values,
    enable_caching=False,
)

print(f"Skip Architecture Search PipelineJob initialized: {skip_job.display_name}")